# R5.A.04 — Qualité algorithmique : Séance 1
## Le détective des tris 🔍

**But Informatique 3e année (alternance) — 4h**

### Consignes distanciel
- Travaillez directement dans ce notebook : il contient le cours, les outils et les exercices.
- Si vous perdez la connexion : continuez en autonomie, tout est ici. Une visio de rattrapage est prévue.
- Sauvegardez régulièrement (`Ctrl+S`). Le livrable final est ce notebook complété + votre rapport d'enquête.
- Légende des difficultés : 🟢 socle (obligatoire) · 🟡 consolidation (si à l'aise) · 🔴 défi (optionnel)

## 1. Accroche : quel code est le meilleur ? 🟢

Voici deux fonctions qui calculent **la moyenne mobile** d'une liste (utile en finance, monitoring, etc.).

```python
# Version A
def moyenne_mobile_A(l, k):
    resultats = []
    for i in range(len(l) - k + 1):
        fenetre = l[i:i+k]
        resultats.append(sum(fenetre) / k)
    return resultats

# Version B
def moyenne_mobile_B(l, k):
    resultats = []
    courant = sum(l[:k])
    for i in range(len(l) - k + 1):
        resultats.append(courant / k)
        if i + k < len(l):
            courant += l[i+k] - l[i]
    return resultats
```

**Question 1** (répondez dans la cellule suivante) : sans exécuter le code, laquelle choisissez-vous et pourquoi ?

**Question 2** : exécutez le benchmark de la cellule d'après, puis comparez. Surprise ?

In [ ]:
# Votre réponse à la Question 1 ici (double-clique pour éditer) :
#
# Question 2 :

In [ ]:
import time
import random

def gen_liste(n):
    """Génère une liste aléatoire de n entiers entre 0 et 10*n."""
    return [random.randint(0, 10 * n) for _ in range(n)]

def mesure(fonction, liste,k):
    """Mesure en secondes le temps d'exécution de `fonction(liste)`.
    Renvoie un tuple (resultat, temps)."""
    debut = time.perf_counter()
    resultat = fonction(liste,k)
    duree = time.perf_counter() - debut
    return resultat, duree

# --- Benchmark de l'accroche ---
l = gen_liste(20_000)
k=100
_, tA = mesure(moyenne_mobile_A, l,k)
_, tB = mesure(moyenne_mobile_B, l,k)
print(f"Version A : {tA:.3f} s")
print(f"Version B : {tB:.3f} s")
print(f"Ratio A/B : {tA/tB:.0f} fois plus lent" if tB > 0 else "")

In [ ]:
def moyenne_mobile_A(l, k):
    resultats = []
    for i in range(len(l) - k + 1):
        fenetre = l[i:i+k]
        resultats.append(sum(fenetre) / k)
    return resultats

def moyenne_mobile_B(l, k):
    resultats = []
    courant = sum(l[:k])
    for i in range(len(l) - k + 1):
        resultats.append(courant / k)
        if i + k < len(l):
            courant += l[i+k] - l[i]
    return resultats

**À retenir** : la qualité algorithmique ne se devine pas, elle se **mesure**. Et un code « qui marche » peut être des centaines de fois plus lent qu'un autre. Dans cette séance, vous allez apprendre à mesurer, puis à *expliquer* ce que vous mesurez avec un outil fondamental : la **complexité asymptotique**.

## 2. La qualité algorithmique : de quoi parle-t-on ? 🟢

Un algorithme est de **qualité** s'il satisfait un compromis entre 4 axes :

| Axe | Question à se poser |
|---|---|
| **Correction** | Renvoie-t-il le bon résultat, dans tous les cas ? |
| **Efficacité temps** | Combien de temps prend-il quand les données grandissent ? |
| **Efficacité mémoire** | Combien de mémoire consomme-t-il ? |
| **Lisibilité / maintenabilité** | Un collègue peut-il le relire et le modifier sans risque ? |

**En entreprise (alternance)** : votre tuteur vous dira rarement « ce code est en O(n²) ». Il dira « c'est trop lent », « ça plante sur les gros fichiers », « on comprend rien à ton code ». Ce module vous donne les outils pour diagnostiquer et *argumenter*.

**Exemple d'accroche métier** : chercher un client dans une base de 10 millions de lignes.
- Parcours complet : ~10 millions d'opérations.
- Dichotomie (si trié) : ~23 opérations. (10 000 000 ≈ 2²³)

C'est LA question du module : **comment mon programme se comporte-t-il quand les données deviennent grandes ?**

## 3. Votre boîte à outils (fourni, ne pas modifier) 🟢

In [ ]:
# Ces fonctions sont fournies. Vous les utiliserez toute la séance.
# (redéfinies ici pour que la cellule soit autonome)

def gen_liste(n):
    return [random.randint(0, 10 * n) for _ in range(n)]

def mesure(fonction, liste):
    debut = time.perf_counter()
    resultat = fonction(liste)
    return resultat, time.perf_counter() - debut

def tableau_mesures(fonction, tailles=(100, 1_000, 10_000)):
    """Mesure `fonction` sur des listes de tailles croissantes et affiche un tableau."""
    print(f"{'taille':>10} | {'temps (s)':>12}")
    print("-" * 26)
    for n in tailles:
        l = gen_liste(n)
        _, duree = mesure(fonction, l)
        print(f"{n:>10} | {duree:>12.6f}")

## 4. Activité 2 — Mesurer les tris mystères 🟢 (20 min)

Vous disposez de 4 algorithmes mystérieux : `tri_A`, `tri_B`, `tri_C`, `tri_D`.
**Vous ne savez pas encore ce qu'ils font précisément.** C'est l'enquête !

**Socle** : exécutez `tableau_mesures` pour chacun des 4 tris (cellule suivante) et notez vos observations dans la cellule markdown en dessous.

🟡 **Consolidation** : ajoutez la taille 50 000. Attention, ça rame — c'est voulu !

🔴 **Défi** : écrivez `mesure_moyenne(fonction, liste, repetitions)` qui répète la mesure et renvoie la moyenne. Pourquoi est-ce plus fiable ?

In [ ]:
def tri_A(l):
    l = l[:]
    for i in range(len(l)):
        m = i
        for j in range(i + 1, len(l)):
            if l[j] < l[m]:
                m = j
        l[i], l[m] = l[m], l[i]
    return l

def tri_B(l):
    l = l[:]
    for i in range(1, len(l)):
        x = l[i]
        j = i
        while j > 0 and l[j - 1] > x:
            l[j] = l[j - 1]
            j -= 1
        l[j] = x
    return l

def tri_C(l):
    l = l[:]
    n = len(l)
    echange = True
    while echange:
        echange = False
        for i in range(n - 1):
            if l[i] > l[i + 1]:
                l[i], l[i + 1] = l[i + 1], l[i]
                echange = True
    return l

def tri_D(l):
    if len(l) <= 1:
        return l[:]
    mi = l[0]
    petits = [x for x in l[1:] if x < mi]
    grands = [x for x in l[1:] if x >= mi]
    return tri_D(petits) + [mi] + tri_D(grands)

for nom, tri in [("tri_A", tri_A), ("tri_B", tri_B),
                 ("tri_C", tri_C), ("tri_D", tri_D)]:
    print(f"=== {nom} ===")
    tableau_mesures(tri, (100, 1_000, 5_000))

**Vos observations** (temps qui explosent ? réguliers ? lesquels semblent les plus rapides ?) :

...votre analyse ici...

## 5. Cours : la complexité asymptotique (Big-O) 🟢

> **Idée clé** : plutôt que de compter les secondes (ça dépend de la machine), on compte les **opérations dominantes** en fonction de la taille des données `n`.

### Les classes à connaître

| Classe | Nom usuel | « Ressenti » : quand n ×10... |
|---|---|---|
| O(1) | constant | ...le temps ne change pas |
| O(log n) | logarithmique | ...le temps ×~1,003 (imperceptible) |
| O(n) | linéaire | ...le temps ×10 |
| O(n log n) | linéarithmique | ...le temps ×~12 |
| O(n²) | quadratique | ...le temps ×100 |
| O(2ⁿ) | exponentiel | ...le temps ×~1000 (le monde s'écroule) |

### La règle du détective 🕵️
Quand la taille ×10, regardez par combien le temps a été multiplié :
- ×10 → O(n)
- ×100 → O(n²)
- à peine plus → O(log n)

### Exemple guidé : compter les opérations
```python
for i in range(len(l)):        # n tours
    for j in range(i+1, len(l)):  # en moyenne n/2 tours imbriqués
        ...                        # 1 comparaison
```
Total ≈ n × n/2 = n²/2 comparaisons → on dit **O(n²)**.

**Pourquoi on « jette » le 1/2 et le +1 ?** Parce qu'on cherche le comportement *quand n tend vers l'infini* : n²/2 et n² grandissent de la même manière. Le Big-O garde seulement le terme dominant.

## 6. Activité 3 — L'enquête ! 🟢 (30 min, binômes)

Dans votre binôme (salon virtuel) :
- **le codeur** lance les mesures (Activité 2 + cellules ci-dessous),
- **l'analyste** lit le code des 4 tris et rédige.

**Rédigez votre rapport d'enquête** dans le pad partagé, pour chaque tri :
1. Son nom probable (cherchez dans vos souvenirs de BUT1/BUT2, ou sur le web : *tri par sélection, tri par insertion, tri à bulles, tri rapide (quicksort)*).
2. Sa complexité, avec **deux preuves** : une preuve par la mesure, une preuve par la lecture du code.
3. Un cas concret d'usage : quand l'utiliser / l'éviter ?

**Indices** :
- `tri_A` cherche un minimum à chaque tour... 
- `tri_C` fait remonter les grands éléments vers la droite comme des bulles...
- Un des tris est **récursif** — lequel ? Cherchez l'appel à lui-même.

### 🟡 Consolidation : listes déjà triées ou inversées
Un bon détective teste aussi les cas particuliers. Comparez les temps des 4 tris sur :
- une liste **déjà triée** (`list(range(n))`),
- une liste **triée en ordre inverse** (`list(range(n, 0, -1))`).

L'un des tris devient-il nettement plus rapide sur une liste déjà triée ? Lequel, et pourquoi ? (Indice : sa boucle `while` s'arrête immédiatement.) C'est le fameux **O(n) dans le meilleur des cas** vs **O(n²) dans le pire**.

In [ ]:
# 🟡 Cellule à compléter : comparez les tris sur listes triées / inversées

## 7. Cours : dépasser O(n²) — diviser pour régner 🟢

Les 4 tris mystères n'ont pas tous la même complexité... mais aucun ne fait mieux qu'environ n² comparaisons sur des données aléatoires, sauf peut-être un ?

La famille des tris **optimisés** (fusion, rapide, tas) atteignent **O(n log n)** grâce au principe **diviser pour régner** :
1. **Diviser** le problème en sous-problèmes (souvent en 2),
2. **Régner** : résoudre les sous-problèmes (récursivement),
3. **Combiner** les résultats.

### Le tri fusion (merge sort), en image

```
[38, 27, 43, 3]                    ← diviser
   [38, 27]      [43, 3]
  [38] [27]     [43] [3]            ← régner (taille 1 = trié !)
   [27, 38]      [3, 43]            ← combiner (fusion de listes triées)
     [3, 27, 38, 43]                ← résultat trié
```

Pourquoi O(n log n) ? On découpe log₂(n) fois, et chaque niveau de fusion coûte O(n). Vous l'implémenterez en séance 2 — aujourd'hui, on le **mesure** seulement :

In [ ]:
def tri_fusion(l):
    if len(l) <= 1:
        return l[:]
    milieu = len(l) // 2
    gauche = tri_fusion(l[:milieu])
    droite = tri_fusion(l[milieu:])
    # fusion des deux moitiés triées
    resultat = []
    i = j = 0
    while i < len(gauche) and j < len(droite):
        if gauche[i] <= droite[j]:
            resultat.append(gauche[i]); i += 1
        else:
            resultat.append(droite[j]); j += 1
    resultat.extend(gauche[i:])
    resultat.extend(droite[j:])
    return resultat

tableau_mesures(tri_fusion, (100, 1_000, 10_000, 50_000))

Comparez avec vos mesures des tris mystères : à 50 000 éléments, l'écart devient spectaculaire. **n log n, c'est des années de gains gratuits à grande échelle.**

## 8. Activité 4 — Tracer et deviner 🟢 (30 min, binômes)

Complétez la cellule suivante pour tracer le temps des 4 tris mystères (et de `sorted` de Python) en fonction de la taille, **en échelle logarithmique**.

**Socle** : le graphe avec les 4 tris. Que remarquez-vous sur les formes des courbes ?

🟡 **Consolidation** : ajoutez `sorted` de Python. Surprise ?

🔴 **Défi** : ajoutez sur le même graphe les courbes théoriques `c1·n²` et `c2·n·log(n)` (choisissez c1, c2 pour qu'elles « collent » visuellement).

In [ ]:
import matplotlib.pyplot as plt

tailles = [100, 500, 1_000, 2_000, 5_000, 10_000]

def courbes(fonctions):  # fonctions = dict nom -> fonction
    temps = {nom: [] for nom in fonctions}
    for n in tailles:
        l = gen_liste(n)
        for nom, f in fonctions.items():
            _, d = mesure(f, l)
            temps[nom].append(d)
    plt.figure(figsize=(8, 5))
    for nom, t in temps.items():
        plt.plot(tailles, t, marker="o", label=nom)
    plt.xscale("log"); plt.yscale("log")
    plt.xlabel("taille de la liste (log)")
    plt.ylabel("temps (s, log)")
    plt.legend(); plt.grid(True, which="both", alpha=0.3)
    plt.title("Temps de tri en fonction de la taille")
    plt.show()

courbes({"A": tri_A, "B": tri_B, "C": tri_C, "D": tri_D})
# 🟡 courbes({"A": tri_A, "B": tri_B, "C": tri_C, "D": tri_D, "sorted": sorted})
# 🔴 à vous : ajouter n**2 et n*log2(n) normalisés

**Astuce du détective** : en échelle log-log, une courbe en O(n²) apparaît comme une droite de pente 2, O(n) une pente 1. Mesurez les pentes à l'œil !

## 9. Activité 5 — Exercices différenciés (25 min, individuel) 🟢

### 🟢 Socle (obligatoire — livrable)

**Ex 1** : classez par ordre de croissance : `n`, `n²`, `log n`, `1000`, `2ⁿ`, `n log n`.

**Ex 2** : donnez la complexité de ces snippets :
```python
(a) for x in l: print(x)
(b) for x in l: for y in l: print(x, y)
(c) x = l[0]
(d) while n > 1: n = n // 2
(e) l.sort() puis for x in l: print(x)   # sort est en O(n log n)
```

**Ex 3** : votre mesure de `tri_A` montre : n=1000 → 0,02 s ; n=10 000 → 2 s. La taille a été multipliée par 10, le temps par 100. Conclusion ?

### 🟡 Consolidation
Écrivez `est_trie(l)` (en O(n)) puis vérifiez que les 4 tris mystères renvoient bien une liste triée, sur plusieurs listes aléatoires. C'est votre premier **test** — la séance sur pytest arrive dans le module.

### 🔴 Défi : le tri stupide (bogosort)
```python
def bogosort(l):
    while not est_trie(l):
        random.shuffle(l)
    return l
```
1. Testez-le sur des listes de taille 3, 4, 5, 6... 7 ? Combien de temps ?
2. Quelle est sa complexité moyenne ? Justifiez avec les mots de la séance (nombre de permutations de n éléments : n!).

In [ ]:
# 🟢 Ex 1 : classez ici (réponse en texte)
# 🟢 Ex 2 : (a) O(...)  (b) O(...)  (c) O(...)  (d) O(...)  (e) O(...)
# 🟢 Ex 3 :

# 🟡
def est_trie(l):
    # à compléter
    ...

# 🔴 bogosort ici

## 10. Quiz de sortie 🟢 (auto-corrigé)
Exécutez la cellule : chaque `assert` doit passer silencieusement. Si un `AssertionError` apparaît, relisez la section 5 !

In [ ]:
def quiz():
    # Q1 : quand n est multiplié par 10, un algo en O(n²) voit son temps multiplié par...
    assert (10*10) == 100
    # Q2 : parmi ces complexités, la MEILLEURE (croît le plus lentement) :
    meilleur = min([1, 2, 3, 2], key=lambda x: x)  # O(1) < O(log n) < O(n) < O(n log n) < O(n²)
    assert meilleur == 1
    # Q3 : dichotomie dans un dictionnaire trié de 1 000 000 de mots ≈ combien d'étapes ?
    import math
    assert math.log2(1_000_000) < 20
    # Q4 : le tri fusion est en...
    assert "O(n log n)" == "O(n log n)"
    # Q5 : le Big-O garde...
    assert "le terme dominant" == "le terme dominant"
    print("✅ Quiz réussi ! Vous êtes officiellement détective d'algorithmes.")

quiz()

## 11. À déposer avant la séance 2
1. Ce notebook complété (activités 2, 4, 5 socle)
2. Le rapport d'enquête de votre binôme (nom + complexité + preuves des 4 tris)

**Prochaine séance** : implémentation du tri fusion, diviser pour régner, et première comparaison « juste » entre tris. Le fil rouge continue !